In [2]:
# Libraries.
import pandas as pd
import numpy as np

In [3]:
# Read in data.
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")
orthologTable = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/orthologTable.txt", sep="\t")
orthologTableIDs = orthologTable[~orthologTable["Mouse gene stable ID"].isna()].loc[:, ["Gene stable ID", "Mouse gene stable ID"]]

In [109]:
gtexEP

,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
Gene stable ID,,,,,,,,,
ENSG00000000003,5.799691,22.906008,18.107586,3.414238,15.745596,23.313591,7.128223,10.594314,protein_coding
ENSG00000000005,0.169146,0.753748,0.284215,0.258039,0.956031,0.018972,0.050229,0.206583,protein_coding
ENSG00000000419,21.387617,40.514694,43.498722,24.609381,25.236029,22.603436,22.147223,37.476429,protein_coding
ENSG00000000457,2.830432,6.585192,6.015115,1.902403,3.748399,4.093346,3.044870,5.295540,protein_coding
ENSG00000000460,1.347589,2.231597,2.198975,0.689768,0.988887,1.240150,0.699461,1.596039,protein_coding
...,...,...,...,...,...,...,...,...,...
ENSG00000310553,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310554,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA
ENSG00000310555,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,lncRNA


In [110]:
emtabEP.sum()

Brain                                                1000000.0
Colon                                                1000000.0
Esophagus                                            1000000.0
Heart                                                1000000.0
Kidney                                               1000000.0
Liver                                                1000000.0
Pancreas                                             1000000.0
Stomach                                              1000000.0
Gene type    protein_codingprotein_codingprotein_codinglncR...
dtype: object

In [111]:
orthologTable

,Gene stable ID,Mouse gene stable ID,Mouse homology type,Mouse Gene-order conservation score
0,ENSG00000198888,ENSMUSG00000064341,ortholog_one2one,50.0
1,ENSG00000198763,ENSMUSG00000064345,ortholog_one2one,75.0
2,ENSG00000198804,ENSMUSG00000064351,ortholog_one2one,100.0
3,ENSG00000198712,ENSMUSG00000064354,ortholog_one2one,100.0
4,ENSG00000228253,ENSMUSG00000064356,ortholog_one2one,100.0
...,...,...,...,...
28009,ENSG00000081692,ENSMUSG00000036819,ortholog_one2one,75.0
28010,ENSG00000157873,ENSMUSG00000022074,ortholog_one2many,0.0
28011,ENSG00000157873,ENSMUSG00000042333,ortholog_one2many,100.0
28012,ENSG00000132676,ENSMUSG00000068921,ortholog_one2one,75.0


In [112]:
orthologTest = tuple(orthologTableIDs.iloc[0, :])
humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

In [113]:
# Euclidean distance. My own methodology.
def euclideanDist(humanOrtholog, mouseOrtholog):
    distSum = 0
    for i in range(0, humanOrtholog.shape[1] - 1):
        distSum += np.square(humanOrtholog.iloc[0, i] - mouseOrtholog.iloc[0, i])
    return np.sqrt(distSum)

In [114]:
euclideanDist(humanOrtholog, mouseOrtholog)

np.float64(44492.547301776765)

In [115]:
# Euclidean Distance. Method using numpy's euclidean distance formula.
np.linalg.norm(np.array(humanOrtholog.iloc[:, :-1]) - np.array(mouseOrtholog.iloc[:, :-1]))

np.float64(44492.547301776765)

In [116]:
# Pearson Distance. My own methodology.
def pearsonDist(humanOrtholog, mouseOrtholog):
    # Calculates the Z-Scores for our vectors, and uses these Z-Score vectors to calculate Pearson Distance. The formula was provided in Piasecka et al. 2012.
    ZxT = ((humanOrtholog.iloc[:, :-1] - np.mean(humanOrtholog.iloc[:, :-1])) / np.std(humanOrtholog.iloc[:, :-1])).T
    Zy = (mouseOrtholog.iloc[:, :-1] - np.mean(mouseOrtholog.iloc[:, :-1])) / np.std(mouseOrtholog.iloc[:, :-1])
    return 1 - ((Zy.dot(ZxT) / ZxT.shape[0]).iloc[0, 0])

In [117]:
pearsonDist(humanOrtholog, mouseOrtholog)

np.float64(0.2094305670992722)

In [118]:
# Pearson Distance is 1 - r, and I found online that r is just the correlation matrix between two dataframes. So I tried that method.
corr = humanOrtholog.iloc[:, :-1].reset_index(drop=True).corrwith(mouseOrtholog.iloc[:, :-1].reset_index(drop=True), method="pearson", axis=1)
1 - corr[0]

np.float64(0.20943055191829985)

In [119]:
# TEC
def TEC(humanOrtholog, mouseOrtholog):
    # Turns the vectors binary. So if the expression is greater than 1, we consider that "expressed."
    humanOrthoBinary = (humanOrtholog.iloc[:, :-1] > 1).iloc[0, :]
    mouseOrthoBinary = (mouseOrtholog.iloc[:, :-1] > 1).iloc[0, :]

    humanOnlyTissueNum = ((humanOrthoBinary ^ mouseOrthoBinary) & humanOrthoBinary).sum()
    mouseOnlyTissueNum = ((mouseOrthoBinary ^ humanOrthoBinary) & mouseOrthoBinary).sum()

    return ((humanOnlyTissueNum / 8) + (mouseOnlyTissueNum / 8)) / 2


In [120]:
# This cell just gets the distance and TEC values, so I can append them to the expression profiles later.
myEuclideanDistArr = []
myPearsonDistArr = []
myTECArr = []
for i in range(0, orthologTableIDs.shape[0]):
    orthologTest = tuple(orthologTableIDs.iloc[i, :])
    humanOrtholog = gtexEP[gtexEP.index.isin(orthologTest)]
    mouseOrtholog = emtabEP[emtabEP.index.isin(orthologTest)]

    if not humanOrtholog.empty and not mouseOrtholog.empty:
        myEuclideanDistArr.append((orthologTest[0], orthologTest[1], euclideanDist(humanOrtholog, mouseOrtholog)))
        myPearsonDistArr.append((orthologTest[0], orthologTest[1], pearsonDist(humanOrtholog, mouseOrtholog)))
        myTECArr.append((orthologTest[0], orthologTest[1], TEC(humanOrtholog, mouseOrtholog)))

In [121]:
myEuclidDistDF = pd.DataFrame(myEuclideanDistArr, columns=["Human ID", "Mouse ID", "EuclidDist"])
myPearDistDF = pd.DataFrame(myPearsonDistArr, columns=["Human ID", "Mouse ID", "PearDist"])
myTECDF = pd.DataFrame(myTECArr, columns=["Human ID", "Mouse ID", "TEC"])

In [122]:
# The next 4 cells merge the expression profile dataframe with each distance and TEC column.
orthologTableEuclid = orthologTable.merge(myEuclidDistDF.loc[:, ["Human ID", "EuclidDist"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPear = orthologTableEuclid.merge(myPearDistDF.loc[:, ["Human ID", "PearDist"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPearTEC = orthologTableEuclidPear.merge(myTECDF.loc[:, ["Human ID", "TEC"]].groupby("Human ID").mean().reset_index(), left_on="Gene stable ID", right_on="Human ID", how="outer").set_index("Human ID")
orthologTableEuclidPearTEC.to_csv("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.csv", index=True)
orthologTableEuclidPearTEC.to_parquet("/Users/andrewhsu/Projects/McNair/data/orthologTableDist.parquet", index=True)

In [34]:
a = pd.read_csv("/Users/andrewhsu/Downloads/emtabExpressionProfile 1(in).csv")

In [37]:
a

,Gene stable ID,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
0,ENSMUSG00000000001,3.491125,12.918058,14.327069,7.289805,7.292047,4.574455,0.890161,5.400340,protein_coding
1,ENSMUSG00000000003,0.000000,0.000000,0.000000,0.005368,0.000000,0.000000,0.000000,0.000000,protein_coding
2,ENSMUSG00000000028,0.223612,1.132353,0.561331,2.268044,0.228048,0.087172,0.018164,0.349117,protein_coding
3,ENSMUSG00000000031,0.256743,0.113783,265.025704,1.737279,0.075890,0.130723,0.167222,0.228744,lncRNA
4,ENSMUSG00000000037,0.122693,0.267582,0.328520,0.077485,0.033373,0.001122,0.000000,0.028755,protein_coding
...,...,...,...,...,...,...,...,...,...,...
43383,ENSMUSG00000109573,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unprocessed_pseudogene
43384,ENSMUSG00000109574,0.033280,0.004708,0.002177,0.001534,0.000838,0.000000,0.000000,0.000000,TEC
43385,ENSMUSG00000109575,0.109874,0.000000,0.000000,0.000907,0.000000,0.000000,0.000000,0.000000,TEC
43386,ENSMUSG00000109576,0.002251,0.000000,0.000000,0.002440,0.001285,0.000000,0.000000,0.000000,TEC


In [35]:
b = pd.read_csv("/Users/andrewhsu/Downloads/emtabExpressionProfile.csv")

In [38]:
b

,Gene stable ID,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
0,ENSMUSG00000000001,3.491125,12.918058,14.327069,7.289805,7.292047,4.574455,0.890161,5.400340,protein_coding
1,ENSMUSG00000000003,0.000000,0.000000,0.000000,0.005368,0.000000,0.000000,0.000000,0.000000,protein_coding
2,ENSMUSG00000000028,0.223612,1.132353,0.561331,2.268044,0.228048,0.087172,0.018164,0.349117,protein_coding
3,ENSMUSG00000000031,0.256743,0.113783,265.025704,1.737279,0.075890,0.130723,0.167222,0.228744,lncRNA
4,ENSMUSG00000000037,0.122693,0.267582,0.328520,0.077485,0.033373,0.001122,0.000000,0.028755,protein_coding
...,...,...,...,...,...,...,...,...,...,...
47526,ENSMUSG00000109574,0.033280,0.004708,0.002177,0.001534,0.000838,0.000000,0.000000,0.000000,TEC
47527,ENSMUSG00000109575,0.109874,0.000000,0.000000,0.000907,0.000000,0.000000,0.000000,0.000000,TEC
47528,ENSMUSG00000109576,0.002251,0.000000,0.000000,0.002440,0.001285,0.000000,0.000000,0.000000,TEC
47529,ENSMUSG00000109577,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,processed_pseudogene


In [58]:
a

,Gene stable ID,Brain,Colon,Esophagus,Heart,Kidney,Liver,Pancreas,Stomach,Gene type
0,ENSMUSG00000000001,3.491125,12.918058,14.327069,7.289805,7.292047,4.574455,0.890161,5.400340,protein_coding
1,ENSMUSG00000000003,0.000000,0.000000,0.000000,0.005368,0.000000,0.000000,0.000000,0.000000,protein_coding
2,ENSMUSG00000000028,0.223612,1.132353,0.561331,2.268044,0.228048,0.087172,0.018164,0.349117,protein_coding
3,ENSMUSG00000000031,0.256743,0.113783,265.025704,1.737279,0.075890,0.130723,0.167222,0.228744,lncRNA
4,ENSMUSG00000000037,0.122693,0.267582,0.328520,0.077485,0.033373,0.001122,0.000000,0.028755,protein_coding
...,...,...,...,...,...,...,...,...,...,...
43383,ENSMUSG00000109573,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,unprocessed_pseudogene
43384,ENSMUSG00000109574,0.033280,0.004708,0.002177,0.001534,0.000838,0.000000,0.000000,0.000000,TEC
43385,ENSMUSG00000109575,0.109874,0.000000,0.000000,0.000907,0.000000,0.000000,0.000000,0.000000,TEC
43386,ENSMUSG00000109576,0.002251,0.000000,0.000000,0.002440,0.001285,0.000000,0.000000,0.000000,TEC


In [73]:
b[b["Gene stable ID"].isin(list(set(b["Gene stable ID"]).difference(a["Gene stable ID"])))]["Gene type"].unique()

<ArrowStringArray>
[nan]
Length: 1, dtype: str

In [66]:
b["Gene stable ID"]

0        ENSMUSG00000000001
1        ENSMUSG00000000003
2        ENSMUSG00000000028
3        ENSMUSG00000000031
4        ENSMUSG00000000037
                ...        
47526    ENSMUSG00000109574
47527    ENSMUSG00000109575
47528    ENSMUSG00000109576
47529    ENSMUSG00000109577
47530    ENSMUSG00000109578
Name: Gene stable ID, Length: 47531, dtype: str